<div align="center">
  <img src="https://raw.githubusercontent.com/NaumanHSA/neurosurfer/main/docs/assets/banner/neurosurfer_banner_white.png" alt="Neurosurfer" width="45%"/>
</div>

<br/>

# 04 — MCP Servers

The **Model Context Protocol (MCP)** is an open standard for connecting LLM apps to external
tools and data. An MCP **server** exposes a set of tools (and resources/prompts); an MCP
**client** connects to it and calls them. The payoff: instead of writing every tool yourself
(notebook [02](02_custom_tools.ipynb)), you plug into the whole MCP ecosystem — GitHub, Slack,
Postgres, the filesystem, Puppeteer, and hundreds more — with **zero tool code**.

Neurosurfer speaks MCP in **two directions**:

| Direction | What it means | Status |
|---|---|---|
| **Client** | Neurosurfer connects to external MCP servers; their tools become normal `Tool`s your agents can call | ✅ available now |
| **Server** | Neurosurfer *hosts* your registered workflows as an MCP server, so Claude Desktop / Cursor / any MCP client can call them | 🚧 coming soon (§9) |

This notebook covers the client today and previews the server.

**You'll learn to:**

1. Configure an MCP server (`McpServerConfig`) and connect with `McpManager`
2. See how a remote tool becomes a Neurosurfer `Tool` (raw JSON Schema + annotation→flag mapping)
3. Call MCP tools directly, then hand them to an `AgenticLoop`
4. Gate untrusted MCP calls with the **MCP permission policy**
5. Persist servers (`mcp.json`) and drive them from the `/mcp` CLI command
6. (Preview) Host your workflows as an MCP server

---

## 1. Setup

MCP support is an **optional extra** — install it with:

```bash
pip install "neurosurfer[mcp]"
```

Same boilerplate as the earlier notebooks: point Python at the repo root, configure the
LM Studio connection, and define a notebook-friendly `IOHandler`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os, json, asyncio, tempfile
from pathlib import Path

# Point Python at the repo root when running from tutorials/
repo_root = Path(os.getcwd()).parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import neurosurfer
print(f"neurosurfer {neurosurfer.__version__}")

# Confirm the MCP extra is installed.
try:
    import mcp
    print("mcp SDK: OK")
except ImportError:
    print("mcp SDK missing — run: pip install 'neurosurfer[mcp]'")

In [ ]:
# ── LM Studio connection ──────────────────────────────────────
# Start LM Studio, load a tool-capable model, and turn the local server ON (port 1234).

LM_STUDIO_URL   = "http://localhost:1234/v1"
LM_STUDIO_MODEL = "google/gemma-4-12b-qat"   # adjust to your loaded model
CONTEXT_WINDOW  = 32_768

from neurosurfer.llm.providers.openai import OpenAICompatProvider

provider = OpenAICompatProvider(
    model          = LM_STUDIO_MODEL,
    base_url       = LM_STUDIO_URL,
    api_key        = "lm-studio",
    context_window = CONTEXT_WINDOW,
)
print(f"Provider: {provider.model}")

In [ ]:
# ── JupyterIO ───────────────────────────────────────────────
# Agents (and the MCP gate) need an IOHandler for approvals. This notebook version
# auto-approves everything and prints what it approved.

class JupyterIO:
    """Auto-approving IOHandler for notebook demos."""

    async def ask(self, question: str, options=None) -> str:
        print(f"  [ask] {question}")
        return "yes"

    async def request_plan_approval(self, plan: str):
        return True, ""

    async def request_shell_approval(self, command: str, reason: str) -> bool:
        print(f"  [approved] {reason}: {command}")
        return True

    async def request_write_approval(self, path: str, summary: str) -> str:
        return "once"

    def notify(self, message: str) -> None:
        print(f"  [notify] {message}")

---

## 2. How MCP fits Neurosurfer

Four pieces, mirroring the provider-profile design you saw for LLMs:

| Piece | Module | Role |
|---|---|---|
| `McpServerConfig` | `neurosurfer.config.mcp` | How to reach one server (stdio command **or** http url) |
| `McpStore` | `neurosurfer.config.mcp` | Persists configured servers to `~/.neurosurfer/mcp.json` |
| `McpManager` | `neurosurfer.mcp` | Connects, discovers tools, and publishes them to the tool pool |
| `McpTool` | `neurosurfer.mcp` | One remote tool wrapped as a Neurosurfer `Tool` |

**How a remote tool becomes a `Tool`:** the server hands Neurosurfer a *raw JSON Schema* for each
tool — so `McpTool` passes that schema straight through (no Pydantic round-trip) and maps the
server's **annotations** onto the behaviour flags the agent loop already understands:

| MCP annotation | Neurosurfer flag |
|---|---|
| `readOnlyHint` | `is_read_only` |
| `destructiveHint` | `is_destructive` |
| `openWorldHint` | folds into `is_concurrency_safe` |

> **Lifecycle note (important for notebooks).** An MCP session owns a subprocess / socket whose
> async context must be **opened and closed in the same task**. In the `neurosurfer` CLI the
> long-lived REPL task handles this for you. In a notebook, each top-level `await` cell is a
> *separate* task — so every live cell below does its own `async with McpManager(...) as mgr:`
> (connect → use → close, all in one cell). Reconnecting is cheap.

---

## 3. A demo MCP server to test against

So the notebook runs end-to-end without any external service, we write a tiny **stdio** MCP
server using the official SDK's `FastMCP` helper. It exposes three tools — two read-only
(`get_weather`, `fx_rate`) and one mutating (`create_note`, to demonstrate the permission gate).

In the real world you'd point at someone else's server instead (see §8) — the client code is
identical.

In [ ]:
MCP_SERVER_SRC = '''"""A tiny MCP server for the Neurosurfer tutorial (stdio transport)."""
from mcp.server.fastmcp import FastMCP

server = FastMCP("tutorial-tools")


@server.tool(annotations={"readOnlyHint": True})
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    table = {"lahore": "34°C, sunny", "london": "12°C, rainy", "tokyo": "19°C, cloudy"}
    return table.get(city.strip().lower(), f"No data for {city}; assume 20°C and clear.")


@server.tool(annotations={"readOnlyHint": True})
def fx_rate(amount: float, base: str, quote: str) -> str:
    """Convert an amount between two currencies using fixed demo rates."""
    rates = {"USD": 1.0, "EUR": 0.92, "GBP": 0.79, "PKR": 278.0}
    b, q = base.upper(), quote.upper()
    if b not in rates or q not in rates:
        return f"Unknown currency. Known: {chr(44).join(rates)}"
    converted = amount / rates[b] * rates[q]
    return f"{amount} {b} = {converted:.2f} {q}"


@server.tool(annotations={"readOnlyHint": False, "destructiveHint": True})
def create_note(title: str, body: str) -> str:
    """Create a note. Mutating — used to demonstrate the permission gate."""
    return f"Saved note {title!r} ({len(body)} chars)."


if __name__ == "__main__":
    server.run("stdio")
'''

# Write it to a temp file the stdio transport can launch.
SERVER_PATH = str(Path(tempfile.gettempdir()) / "ns_tutorial_mcp_server.py")
Path(SERVER_PATH).write_text(MCP_SERVER_SRC, encoding="utf-8")
print("wrote", SERVER_PATH)

---

## 4. Connect & discover

`McpServerConfig` describes the connection; `McpManager` opens it and lists the tools. A
**stdio** server is launched as a child process — we run it with the current Python interpreter.

Everything that needs the live connection happens inside the `async with` block.

In [ ]:
from neurosurfer.config.mcp import McpServerConfig
from neurosurfer.mcp import McpManager
from neurosurfer.tools import ToolContext

cfg = McpServerConfig(
    name      = "tutorial",
    transport = "stdio",
    command   = sys.executable,   # launch the demo server with this Python
    args      = [SERVER_PATH],
)

async with McpManager([cfg]) as mgr:
    # 1. Connection status (per server).
    for st in mgr.status():
        print(f"server {st.name!r}: connected={st.connected}, tools={st.tools}")

    # 2. Each remote tool is now a Neurosurfer Tool, with flags from its annotations.
    print("\nDiscovered tools:")
    for t in mgr.tools():
        ro = t.is_read_only(t.parse_args({}))
        print(f"  · {t.name:12} read_only={ro!s:5}  — {t.description.splitlines()[0]}")

    # 3. The schema came straight from the server (no Pydantic round-trip).
    weather = next(t for t in mgr.tools() if t.name == "get_weather")
    print("\nget_weather input schema:")
    print(json.dumps(weather.schema.input_schema, indent=2))

    # 4. Call a tool directly — no agent, just tool.run(args, ctx).
    ctx = ToolContext(cwd=repo_root, io=JupyterIO())
    res = await weather.run({"city": "Tokyo"}, ctx)
    print("\nDirect call → get_weather(Tokyo):", res.content)

---

## 5. Give MCP tools to an agent

MCP tools are ordinary `Tool`s, so they drop straight into a `ToolPool`. Here we build a pool
from the discovered tools plus `finish`, and let an `AgenticLoop` decide which to call.

> **Global option.** Because `McpManager` also publishes discovered tools to the live registry,
> `default_pool()` / `all_tools()` include them automatically while the manager is connected —
> so agents built the usual way see MCP tools with no extra wiring. We pass them explicitly
> below just to make the pool obvious.

The agent run must stay **inside** the `async with` block — the tools hold the live session.

In [ ]:
from neurosurfer.agents import AgenticLoop, Guardrails, events
from neurosurfer.tools import ToolPool
from neurosurfer.tools.builtin import FinishTool

task = "What's the weather in London, and how much is 100 USD in PKR?"

async with McpManager([cfg]) as mgr:
    pool = ToolPool([*mgr.tools(), FinishTool()])
    print("Agent tool pool:", pool.names(), "\n" + "─" * 60)

    agent = AgenticLoop(
        provider      = provider,
        tools         = pool,
        system_prompt = (
            "You are a helpful assistant. Use the get_weather and fx_rate tools to answer. "
            "Call finish when done."
        ),
        guardrails    = Guardrails(max_turns=8),
        io            = JupyterIO(),
        cwd           = repo_root,
    )

    # verbose=True (default) prints the live Thinking…/Running… trace; we render the answer.
    async for ev in agent.run(task):
        if isinstance(ev, events.TextDelta):
            print(ev.text, end="", flush=True)

---

## 6. Permissions — the MCP gate

External tools are untrusted by default. A `Guardrails.mcp_policy` controls them:

| Policy | Behaviour |
|---|---|
| `"gated"` *(default)* | Read-only tools pass; anything that may mutate asks for approval |
| `"open"` | All MCP calls allowed without asking |
| `"denied"` | No MCP calls at all |

The gate reads each tool's `readOnlyHint`, so `get_weather` (read-only) is never questioned,
while `create_note` (mutating) is. Below we check the gate directly with a **denying** IO.

In [ ]:
from neurosurfer.agents.runtime.permissions import Guardrails, Permissions

class DenyingIO(JupyterIO):
    async def request_shell_approval(self, command: str, reason: str) -> bool:
        print(f"  [DENIED] {reason}: {command}")
        return False

async with McpManager([cfg]) as mgr:
    weather = next(t for t in mgr.tools() if t.name == "get_weather")
    note    = next(t for t in mgr.tools() if t.name == "create_note")
    ctx_ok  = ToolContext(cwd=repo_root, io=JupyterIO())   # approves
    ctx_no  = ToolContext(cwd=repo_root, io=DenyingIO())   # denies

    def gate(policy, tool, ctx):
        perms = Permissions(Guardrails(mcp_policy=policy), repo_root)
        return perms.check(tool.name, tool.parse_args({}), ctx, "default", tool=tool)

    print("gated  + read-only get_weather :", (await gate("gated",  weather, ctx_no)).allow, "(never asked)")
    print("gated  + mutating  create_note :", (await gate("gated",  note,    ctx_ok)).allow, "(asked → approved)")
    print("gated  + mutating  (denied IO) :", (await gate("gated",  note,    ctx_no)).allow)
    print("open   + mutating  create_note :", (await gate("open",   note,    ctx_no)).allow, "(no prompt)")
    print("denied + read-only get_weather :", (await gate("denied", weather, ctx_ok)).allow)

---

## 7. Persisting servers + the `/mcp` CLI

`McpStore` saves your servers to `~/.neurosurfer/mcp.json` (mode `0600` — headers can carry
tokens). Values like `${GITHUB_TOKEN}` in `env`/`headers` are expanded from the environment at
connect time, so secrets never live in the file. Below we use a throwaway path.

In [ ]:
from neurosurfer.config.mcp import McpStore

store = McpStore(path=Path(tempfile.gettempdir()) / "demo_mcp.json")
store.add(McpServerConfig(name="tutorial", transport="stdio",
                          command=sys.executable, args=[SERVER_PATH]))
store.add(McpServerConfig(name="github", transport="http",
                          url="https://api.githubcopilot.com/mcp/",
                          headers={"Authorization": "Bearer ${GITHUB_TOKEN}"},
                          enabled=False))

for s in store.list():
    print(s.summary())

In the **`neurosurfer` CLI** the same store powers a `/mcp` command, and connected servers'
tools are available to every agent automatically:

```text
/mcp list                 # configured servers + live status
/mcp add                  # interactive: name, transport, command/url
/mcp tools tutorial       # introspect a server's tools
/mcp enable  <name>
/mcp disable <name>
/mcp remove  <name>
/mcp reconnect            # re-read config and reconnect
```

`neurosurfer doctor` also reports MCP reachability and tool counts.

---

## 8. Real-world MCP servers

The demo server is local, but the client code is the same for any server. A few popular ones
(these need `npx`/Docker and possibly an API token — shown as config, not executed here):

```python
# Filesystem (stdio via npx)
McpServerConfig(
    name="filesystem", transport="stdio",
    command="npx",
    args=["-y", "@modelcontextprotocol/server-filesystem", "/path/to/dir"],
)

# GitHub (remote http, token from env)
McpServerConfig(
    name="github", transport="http",
    url="https://api.githubcopilot.com/mcp/",
    headers={"Authorization": "Bearer ${GITHUB_TOKEN}"},
)
```

Add either to your `McpStore` (or via `/mcp add`) and its tools appear in your agents' pool.

---

## 9. (Coming soon) Hosting workflows as an MCP server

> 🚧 **Preview — not yet implemented.** This is the second MCP direction on the roadmap
> (see `docs/plans/INTEGRATIONS_PLAN.md`, Phase 2). The snippet below shows the *planned*
> interface; this section will become runnable when the feature lands.

The idea: take the workflows you build with the Architect (notebook [03](03_graph_agents.ipynb))
and expose each one as an MCP **tool**, so any MCP client — Claude Desktop, Cursor, another
Neurosurfer — can call your workflow natively.

```bash
# Planned CLI (Phase 2)
neurosurfer serve --mcp            # serve registered workflows over stdio (for desktop clients)
neurosurfer serve --mcp-http       # ...or over streamable HTTP
```

```python
# Planned Python API (Phase 2) — illustrative, will change
# from neurosurfer.mcp.server import serve_workflows
# serve_workflows(provider=provider, transport="stdio")
```

Each registered workflow becomes one MCP tool: its declared inputs become the tool's input
schema, and a call runs the workflow and returns its final artifact. Workflows run under a
restrictive guardrail set by default, since the caller is remote.

<!-- TODO(P2): replace this section with runnable code once `neurosurfer serve --mcp` ships. -->

---

## Summary

| Step | API |
|---|---|
| Describe a server | `McpServerConfig(name, transport, command/args or url/headers)` |
| Persist servers | `McpStore` → `~/.neurosurfer/mcp.json` (or `/mcp add`) |
| Connect & discover | `async with McpManager([cfg]) as mgr: mgr.tools()` |
| Inspect a tool | `tool.schema`, `tool.is_read_only(...)` (from MCP annotations) |
| Call directly | `await tool.run(args, ctx)` |
| Use with an agent | `ToolPool([*mgr.tools(), FinishTool()])` → `AgenticLoop` |
| Gate external calls | `Guardrails(mcp_policy="gated" | "open" | "denied")` |

**Key ideas**
- MCP turns *someone else's* tools into Neurosurfer `Tool`s — no tool code to write.
- Schemas pass through raw; annotations drive the read-only / destructive / concurrency flags.
- External tools are **gated** by default (`mcp_policy="gated"`): read-only passes, mutating asks.
- Open/close a session in the **same task** — one `async with` per notebook cell; the CLI handles it for you.
- Hosting workflows as an MCP server is coming next (§9).

### What's next

- **[02 — Custom Tools](02_custom_tools.ipynb):** write your own `Tool`s when no MCP server exists.
- **[03 — Graph Agents](03_graph_agents.ipynb):** build the workflows you'll soon be able to serve over MCP.
- **[05 — Capstone: The Insight Engine](05_capstone_insight_engine.ipynb):** see MCP tools join function + vision nodes in one report-writing graph.